# Virtual Try-On — Model Comparison Pilot (V1)
Person image + garment image → person wearing the garment, across every viable
hosted arm, vs the **Qwen 2511** baseline. Budget ceiling: **$10**.

## ⚙️ One-time setup (do this first)
1. **fal API key** — two options, either works:
   - *Recommended:* click the **🔑 key icon** in Colab's left sidebar →
     **Add new secret** → name `FAL_KEY`, value = your fal key
     (`xxxxxxxx:xxxxxxxx` format) → toggle **Notebook access ON**.
   - *Or do nothing:* §2 will prompt you to paste the key (hidden input).
2. **Runtime** — plain CPU is fine (all models are hosted APIs; no GPU needed).
3. **Nothing to upload** — §2 clones the repo; the 30+30 test set comes with it.
4. Google Drive will ask for mount permission (run outputs are saved there).

## ▶️ Run flow
Run all (free cells) → flip `RUN_TRIAGE=True` in §1 (~$1.30) → eyeball grids,
set `SURVIVORS` → flip `RUN_GRID=True` → **score outputs at §8 (your judging
station)** → leaderboard §10 → verdict §11.
**Paid cells never run on "Run all"** — flags gate every dollar.

Docs: `prd/V1_PILOT.md` (scope) · `execution_conventions.md` (conventions).

In [ ]:
#  §1 · Settings ------------------------------------------------------------
SEED_BASE = 46            # per-pair seed = SEED_BASE + pair index (fixed across arms)
PROMPT_TEMPLATE = (
    "Replace the clothing of the person in image 1 with the garment shown in "
    "image 2. Keep the person's face, hair, pose, hands, body and the "
    "background completely unchanged. Preserve the garment's exact color, "
    "pattern, print, and cut."
)
RUN_TRIAGE  = False       # 💰 arms × 4 pairs × 1 gen  (~$1.30)
RUN_GRID    = False       # 💰 SURVIVORS × 12 pairs × 1 gen
RUN_RESERVE = False       # 💰 TOP2 × 12 pairs × 1 extra gen (best-of-2)
SURVIVORS = []            # fill after judging triage, e.g. ["qwen_2511", "fashn_v16", ...]
TOP2      = []            # fill after judging grid
BUDGET_USD = 10.00
ARMS_ENABLED = {          # flip any arm off if it misbehaves
    "qwen_2511": True,       # baseline — always on
    "klein_4b_edit": True,
    "flux_vto_v1": True,     # v2 exists only on BFL's direct API, not fal
    "fashn_v16": True,
    "qwen_image3_edit": True,
    "klein_tryon_lora": True,
    "seedream5_lite": True,
}
print(f"arms on: {[k for k,v in ARMS_ENABLED.items() if v]}")

In [ ]:
#  §2 · Setup (free) --------------------------------------------------------
# fal client + key from Colab Secrets + test set from GitHub
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fal-client", "pillow", "pandas", "matplotlib", "jsonschema"], check=True)
try:                                   # option 1: Colab Secrets (🔑 sidebar)
    from google.colab import userdata
    os.environ["FAL_KEY"] = userdata.get("FAL_KEY")
except Exception:                      # option 2: paste it (hidden input)
    from getpass import getpass
    os.environ["FAL_KEY"] = getpass("Paste your fal API key (input hidden): ").strip()
assert os.environ["FAL_KEY"], "no FAL_KEY provided"   # never printed
REPO_URL = "https://github.com/101011101/magichour_takehome.git"   # ← filled at build time
REPO_DIR = "/content/tryon_repo"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = "/content/drive/MyDrive/tryon_pilot_runs"
os.makedirs(OUT_ROOT, exist_ok=True)
print("setup ok:", REPO_DIR, "→ outputs:", OUT_ROOT)

In [ ]:
#  §3 · Test set (free) -----------------------------------------------------
import pandas as pd
TS = os.path.join(REPO_DIR, "test_set")
pairs = pd.read_csv(os.path.join(TS, "pairs.csv"))
# deterministic subsets: triage = 1 easy + 2 medium + 1 hard; grid = stratified 12
tri = pd.concat([pairs[pairs.difficulty == "easy"].head(1),
                 pairs[pairs.difficulty == "medium"].head(2),
                 pairs[pairs.difficulty == "hard"].head(1)])
gri = pd.concat([pairs[pairs.difficulty == "easy"].head(2),
                 pairs[pairs.difficulty == "medium"].head(5),
                 pairs[pairs.difficulty == "hard"].head(5)])
TRIAGE_PAIRS = list(tri.itertuples(index=False))
GRID_PAIRS   = list(gri.itertuples(index=False))
def img_path(kind, pid): return os.path.join(TS, kind, f"{pid}.jpg")
print(f"{len(pairs)} pairs loaded — triage {len(TRIAGE_PAIRS)}, grid {len(GRID_PAIRS)}")

In [ ]:
#  §4 · Preflight + cost estimate (free) ------------------------------------
import fal_client
assert os.environ.get("FAL_KEY"), "FAL_KEY missing — add it in Colab Secrets (🔑 sidebar)"
for _, pid, gid in [(0, r.person_id, r.garment_id) for r in TRIAGE_PAIRS]:
    for p in (img_path("people", pid), img_path("garments", gid)):
        assert os.path.exists(p), f"missing test image: {p}"
n_on = sum(ARMS_ENABLED.values())
est = {"triage": n_on * len(TRIAGE_PAIRS) * 0.045,
       "grid":   max(len(SURVIVORS), 4) * len(GRID_PAIRS) * 0.045,
       "reserve": 2 * len(GRID_PAIRS) * 0.045}
print("est. spend if all stages run: "
      f"${sum(est.values()):.2f} of ${BUDGET_USD:.2f} — {est}")

In [ ]:
#  §5 · Harness: arm registry + try_on() (free) -----------------------------
import io, json, time, hashlib, requests
from PIL import Image

# All endpoint IDs + schemas verified against live fal.ai /api pages (2026-08-08).
# Note: FLUX VTO **v2** is NOT on fal (BFL direct API only) — v1 used here.
def _std(person_url, garment_url, seed):        # multi-image instruction editors
    return {"prompt": PROMPT_TEMPLATE, "image_urls": [person_url, garment_url],
            "seed": seed, "num_images": 1}
ARM_REGISTRY = {
    "qwen_2511":       {"endpoint": "fal-ai/qwen-image-edit-2511",   # $0.03/MP
                        "args": _std, "est_usd": 0.03},
    "klein_4b_edit":   {"endpoint": "fal-ai/flux-2/klein/4b/distilled/edit",
                        "args": _std, "est_usd": 0.015},             # $0.014+0.001/MP out
    "flux_vto_v1":     {"endpoint": "fal-ai/flux-pro/v1/vto",        # ~$0.0475 @1MP
                        # dedicated params; person ≤2MP (≤1MP rec.), garment ≤1MP (≤0.5MP rec.)
                        "args": lambda p, g, s: {"prompt": PROMPT_TEMPLATE,
                                                 "human_image_url": p,
                                                 "garment_image_url": g, "seed": s},
                        "resize": {"person_mp": 1.0, "garment_mp": 0.5},
                        "est_usd": 0.048},
    "fashn_v16":       {"endpoint": "fal-ai/fashn/tryon/v1.6",       # flat $0.075/gen
                        # no prompt param; fixed 864x1296 output
                        "args": lambda p, g, s: {"model_image": p, "garment_image": g,
                                                 "category": "auto", "mode": "quality",
                                                 "garment_photo_type": "auto",
                                                 "seed": s, "num_samples": 1},
                        "est_usd": 0.075},
    "qwen_image3_edit":{"endpoint": "alibaba/qwen-image-3/edit",     # $0.04 @1K
                        # prompt-expansion is ON by default → disable for fairness
                        "args": lambda p, g, s: {**_std(p, g, s),
                                                 "enable_prompt_expansion": False},
                        "est_usd": 0.04},
    "klein_tryon_lora":{"endpoint": "fal-ai/flux-2-lora-gallery/virtual-tryon",
                        # $0.021/compute-second; LoRA trigger word required
                        "args": lambda p, g, s: {"prompt": "TRYON " + PROMPT_TEMPLATE,
                                                 "image_urls": [p, g], "seed": s,
                                                 "num_images": 1},
                        "est_usd": 0.04},
    "seedream5_lite":  {"endpoint": "fal-ai/bytedance/seedream/v5/lite/edit",
                        # ⚠️ no seed input — reproducibility not controllable on this arm
                        "args": lambda p, g, s: {"prompt": PROMPT_TEMPLATE,
                                                 "image_urls": [p, g], "num_images": 1},
                        "est_usd": 0.035},
}
_upload_cache, RUNS = {}, []
def upload(path, max_mp=None):
    """Upload to fal, optionally downscaling to a megapixel cap (FLUX VTO limits)."""
    key = (path, max_mp)
    if key not in _upload_cache:
        src = path
        if max_mp:
            im = Image.open(path); mp = im.width * im.height / 1e6
            if mp > max_mp:
                sc = (max_mp / mp) ** 0.5
                im = im.resize((int(im.width * sc), int(im.height * sc)))
                src = f"/tmp/{hashlib.md5(str(key).encode()).hexdigest()}.jpg"
                im.convert("RGB").save(src, quality=95)
        _upload_cache[key] = fal_client.upload_file(src)
    return _upload_cache[key]
def try_on(arm, pair, seed, stage):
    cfg = ARM_REGISTRY[arm]
    rz = cfg.get("resize", {})
    t0 = time.time()
    args = cfg["args"](upload(img_path("people", pair.person_id), rz.get("person_mp")),
                       upload(img_path("garments", pair.garment_id), rz.get("garment_mp")),
                       seed)
    try:
        res = fal_client.subscribe(cfg["endpoint"], arguments=args)
    except Exception as e:
        RUNS.append({"arm": arm, "pair": f"{pair.person_id}x{pair.garment_id}",
                     "stage": stage, "seed": seed, "ok": False, "err": str(e)[:200]})
        return None
    url = (res.get("images") or [res.get("image", {})])[0].get("url")
    img = Image.open(io.BytesIO(requests.get(url).content)).convert("RGB")
    rid = f"{stage}_{arm}_{pair.person_id}x{pair.garment_id}_s{seed}"
    rdir = os.path.join(OUT_ROOT, rid); os.makedirs(rdir, exist_ok=True)
    img.save(os.path.join(rdir, "result.png"))
    rec = {"arm": arm, "pair": f"{pair.person_id}x{pair.garment_id}", "stage": stage,
           "seed": seed, "ok": True, "latency_s": round(time.time() - t0, 1),
           "est_usd": cfg["est_usd"], "endpoint": cfg["endpoint"], "dir": rdir}
    json.dump({**rec, "args_keys": list(args)}, open(os.path.join(rdir, "run_config.json"), "w"), indent=2)
    RUNS.append(rec)
    return img
def spent(): return sum(r.get("est_usd", 0) for r in RUNS if r.get("ok"))

In [ ]:
#  §6a · 💰 TRIAGE RUN — flips on with RUN_TRIAGE=True ----------------------
RESULTS = globals().get("RESULTS", {})   # (stage, arm, pair, seed) -> Image
if RUN_TRIAGE:
    for arm in [a for a, on in ARMS_ENABLED.items() if on]:
        for i, pair in enumerate(TRIAGE_PAIRS):
            if spent() >= BUDGET_USD: raise SystemExit("budget ceiling hit")
            img = try_on(arm, pair, SEED_BASE + i, "triage")
            if img: RESULTS[("triage", arm, i, SEED_BASE + i)] = img
    print(f"triage done — est ${spent():.2f} spent, {len(RESULTS)} ok")
else:
    print("triage skipped (RUN_TRIAGE=False)")

In [ ]:
#  §6b · 💰 GRID RUN — set SURVIVORS first, then RUN_GRID=True --------------
if RUN_GRID:
    assert SURVIVORS, "set SURVIVORS in §1 after judging the triage grid"
    for arm in SURVIVORS:
        for i, pair in enumerate(GRID_PAIRS):
            if spent() >= BUDGET_USD: raise SystemExit("budget ceiling hit")
            img = try_on(arm, pair, SEED_BASE + 100 + i, "grid")
            if img: RESULTS[("grid", arm, i, SEED_BASE + 100 + i)] = img
    print(f"grid done — est ${spent():.2f} spent")
else:
    print("grid skipped (RUN_GRID=False)")
# (§6c reserve: same shape, TOP2 × GRID_PAIRS × seed+200, RUN_RESERVE flag.)

In [ ]:
#  §7 · Comparison grids (free, thumbnails only) ----------------------------
import matplotlib.pyplot as plt
def show_stage(stage, pairs_list, thumb=224):
    arms = sorted({k[1] for k in RESULTS if k[0] == stage})
    if not arms: print(f"no {stage} results yet"); return
    rows = len(pairs_list); cols = len(arms) + 2
    fig, ax = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.4 * rows))
    for r, pair in enumerate(pairs_list):
        ins = [Image.open(img_path("people", pair.person_id)),
               Image.open(img_path("garments", pair.garment_id))]
        for c, (ttl, im) in enumerate(zip(["person", "garment"], ins)):
            ax[r][c].imshow(im.copy().resize((thumb, thumb))); ax[r][c].set_title(ttl, fontsize=7)
        for c, arm in enumerate(arms, start=2):
            hit = next((v for k, v in RESULTS.items()
                        if k[0] == stage and k[1] == arm and k[2] == r), None)
            if hit: ax[r][c].imshow(hit.copy().resize((thumb, thumb)))
            ax[r][c].set_title(arm, fontsize=7)
        for c in range(cols): ax[r][c].axis("off")
    plt.tight_layout(); plt.show()
show_stage("triage", TRIAGE_PAIRS)
show_stage("grid", GRID_PAIRS)

---
# 🔴 §8 · HUMAN JUDGING — **this is your station, Ray**
You judge **twice**:
1. **After triage** (grids above): no scoring — just decide which arms are
   *categorically broken* (ignores garment, mangles person, API dead) and set
   `SURVIVORS = [...]` in §1, then flip `RUN_GRID=True` and re-run §6b→§7.
2. **After the grid run**: score every output below. Run the next cell — it
   builds a blank sheet. For each row give 1–5 on the four criteria
   (5 = flawless):
   - `garment` — is it *the* garment (color/print/cut)?
   - `identity` — same face, hair, body?
   - `scene` — pose + background untouched?
   - `clean` — free of artifacts (hands, seams, plastic skin)?
   Edit the numbers straight in the printed dataframe cell (or open
   `judging_sheet.csv` in the Colab file editor), then run the **save** cell.

In [ ]:
#  §8a · Build judging sheet (free) -----------------------------------------
sheet = pd.DataFrame([{"arm": k[1], "pair_idx": k[2],
                       "pair": f"{GRID_PAIRS[k[2]].person_id}×{GRID_PAIRS[k[2]].garment_id}",
                       "garment": None, "identity": None, "scene": None, "clean": None}
                      for k in sorted(RESULTS) if k[0] == "grid"])
sheet.to_csv("judging_sheet.csv", index=False)
print(f"{len(sheet)} outputs to judge → judging_sheet.csv (person×garment rows)")
# Display one large row at a time to judge comfortably:
def judge_view(row_i, size=380):
    k = sorted([k for k in RESULTS if k[0] == "grid"])[row_i]
    pair = GRID_PAIRS[k[2]]
    fig, ax = plt.subplots(1, 3, figsize=(9, 3.4))
    for a, (ttl, im) in zip(ax, [("person", Image.open(img_path("people", pair.person_id))),
                                 ("garment", Image.open(img_path("garments", pair.garment_id))),
                                 (k[1], RESULTS[k])]):
        a.imshow(im.copy().resize((size, size))); a.set_title(ttl, fontsize=9); a.axis("off")
    plt.show()
# judge_view(0)  # ← step through 0..N-1 as you fill the sheet

In [ ]:
#  §8b · Save judgments (run when sheet is filled) --------------------------
done = pd.read_csv("judging_sheet.csv")
missing = done[["garment", "identity", "scene", "clean"]].isna().sum().sum()
assert missing == 0, f"{int(missing)} blank scores remain — finish the sheet first"
done.to_csv(os.path.join(OUT_ROOT, "judgments.csv"), index=False)
print("judgments saved →", os.path.join(OUT_ROOT, "judgments.csv"))

## §9 · VLM judge — code present, **not run in V1** (`JUDGE_MODEL = None`)
Same rubric as §8, scored by a vision LLM, blind to arm names. Enable later by
setting `JUDGE_MODEL` to e.g. `"gemini-2.5-flash-lite"` / `"claude-haiku-4-5"`
and adding the provider key to Colab Secrets. Verdicts are schema-validated
with up to 3 self-correct retries (pattern from MagicHourOptimize).

In [ ]:
#  §9 · VLM judge (free while JUDGE_MODEL is None) --------------------------
JUDGE_MODEL = None
JUDGE_SCHEMA = {"type": "object", "additionalProperties": False,
                "required": ["garment", "identity", "scene", "clean", "note"],
                "properties": {**{k: {"type": "integer", "minimum": 1, "maximum": 5}
                                  for k in ["garment", "identity", "scene", "clean"]},
                               "note": {"type": "string", "maxLength": 300}}}
JUDGE_PROMPT = (
    "You are judging a virtual try-on output. Image 1: the original person. "
    "Image 2: the reference garment. Image 3: the generated result. Score 1-5 "
    "(5 flawless): garment = is the output garment exactly the reference "
    "(color, print, cut); identity = same face/hair/body as image 1; scene = "
    "pose and background unchanged; clean = free of AI artifacts (hands, seams, "
    "textures). Return ONLY JSON matching the schema.")
def _call_judge(model, images, prompt):
    raise NotImplementedError(
        "wire provider here when a judge is picked: "
        "gemini → google-genai client; claude → anthropic client; both ~10 lines")
def vlm_judge(person_p, garment_p, result_img, model=None, attempts=3):
    from jsonschema import validate, ValidationError
    model = model or JUDGE_MODEL
    if model is None: return None                     # V1: judge is off
    prompt = JUDGE_PROMPT
    for i in range(attempts):
        raw = _call_judge(model, [person_p, garment_p, result_img], prompt)
        try:
            start, end = raw.find("{"), raw.rfind("}") + 1   # extract_json pattern
            verdict = json.loads(raw[start:end]); validate(verdict, JUDGE_SCHEMA)
            return verdict
        except (ValueError, ValidationError) as e:           # self-correct retry
            prompt = (f"{JUDGE_PROMPT}\n\nYour previous reply failed validation "
                      f"({e}). Previous reply:\n{raw}\nReturn corrected JSON only.")
    return None
print("VLM judge loaded — dormant (JUDGE_MODEL=None); human judging is §8")

In [ ]:
#  §10 · Leaderboard (free) -------------------------------------------------
jpath = os.path.join(OUT_ROOT, "judgments.csv")
if os.path.exists(jpath):
    J = pd.read_csv(jpath)
    J["overall"] = J[["garment", "identity", "scene", "clean"]].mean(axis=1)
    board = (J.groupby("arm")[["garment", "identity", "scene", "clean", "overall"]]
               .mean().round(2).sort_values("overall", ascending=False))
    base = J[J.arm == "qwen_2511"].set_index("pair_idx")["overall"]
    wins = {arm: int((g.set_index("pair_idx")["overall"] > base.reindex(g.pair_idx).values).sum())
            for arm, g in J.groupby("arm") if arm != "qwen_2511"}
    print(board.to_string()); print("\npair wins vs qwen_2511:", wins)
else:
    print("no judgments.csv yet — finish §8")

In [ ]:
#  §11 · Verdict card + spend (free) ----------------------------------------
ok = [r for r in RUNS if r.get("ok")]; err = [r for r in RUNS if not r.get("ok")]
print("=" * 46, f"\nruns ok {len(ok)} | failed {len(err)} | est spend ${spent():.2f} / ${BUDGET_USD:.2f}")
if err: print("failures:", *[f"  {e['arm']}: {e['err'][:80]}" for e in err], sep="\n")
if os.path.exists(jpath) and len(board):
    w = board.index[0]
    print(f"leader: {w} (overall {board.loc[w,'overall']}) — "
          f"{'BEATS' if w != 'qwen_2511' else 'baseline still leads'} vs qwen_2511")
print("=" * 46)